# Video-RAG on Kaggle

Ноутбук для запуска текущего проекта `ai-chemp-cu-itmo` в среде Kaggle:
1. поднимает проект в `/kaggle/working`
2. ставит зависимости
3. копирует видео из Kaggle Input
4. строит векторную БД
5. выполняет CLI-поиск

> Рекомендуется включить **Internet = ON** хотя бы для первого прогона, чтобы скачать модели в кэш.

In [ ]:
import os
import shutil
from pathlib import Path

# ---- User config ----
# Укажите корневую папку в /kaggle/input, где лежит ваш репозиторий ai-chemp-cu-itmo
CODE_DATASET_DIR = Path('/kaggle/input/ai-chemp-cu-itmo')
# Укажите папку с исходными видео
VIDEO_INPUT_DIR = Path('/kaggle/input/shrek-dataset')

PROJECT_NAME = 'ai-chemp-cu-itmo'
WORKDIR = Path('/kaggle/working')
PROJECT_DIR = WORKDIR / PROJECT_NAME

# Кэш моделей (переживает перезапуск только в рамках текущего notebook session)
HF_CACHE_DIR = WORKDIR / '.model_cache' / 'hf'
WHISPER_CACHE_DIR = WORKDIR / '.model_cache' / 'whisper'

HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
WHISPER_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('WORKDIR:', WORKDIR)
print('PROJECT_DIR:', PROJECT_DIR)
print('VIDEO_INPUT_DIR:', VIDEO_INPUT_DIR)
print('HF_CACHE_DIR:', HF_CACHE_DIR)
print('WHISPER_CACHE_DIR:', WHISPER_CACHE_DIR)


In [ ]:
# Поднять код проекта в /kaggle/working
if PROJECT_DIR.exists():
    print('Project already exists in working dir:', PROJECT_DIR)
else:
    candidate_a = CODE_DATASET_DIR / PROJECT_NAME
    candidate_b = CODE_DATASET_DIR

    if candidate_a.exists():
        src = candidate_a
    elif candidate_b.exists() and (candidate_b / 'build_vectordb.py').exists():
        src = candidate_b
    else:
        raise FileNotFoundError(
            f'Cannot find project code under {CODE_DATASET_DIR}. '
            'Attach dataset with repository files and update CODE_DATASET_DIR.'
        )

    shutil.copytree(src, PROJECT_DIR)
    print('Copied project to:', PROJECT_DIR)

%cd /kaggle/working/ai-chemp-cu-itmo
!ls -la

In [ ]:
# Установка зависимостей
%cd /kaggle/working/ai-chemp-cu-itmo
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt

# ffmpeg обычно уже есть в Kaggle образе, но проверим
!which ffmpeg && ffmpeg -version | head -n 1

In [ ]:
# Копируем видео в data/ проекта
%cd /kaggle/working/ai-chemp-cu-itmo
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)

video_exts = {'.mp4', '.mkv'}
found = [p for p in VIDEO_INPUT_DIR.rglob('*') if p.suffix.lower() in video_exts]
if not found:
    raise FileNotFoundError(f'No video files (.mp4/.mkv) found in {VIDEO_INPUT_DIR}')

for src in found:
    dst = data_dir / src.name
    if not dst.exists():
        shutil.copy2(src, dst)

print('Copied videos:', [p.name for p in data_dir.iterdir() if p.suffix.lower() in video_exts])

In [ ]:
# Построение векторной БД
%cd /kaggle/working/ai-chemp-cu-itmo
!python build_vectordb.py \
  --data_dir data \
  --output_dir output \
  --db_dir db \
  --device cpu \
  --whisper_model base \
  --hf_cache_dir /kaggle/working/.model_cache/hf \
  --whisper_cache_dir /kaggle/working/.model_cache/whisper

In [ ]:
# Простой CLI-поиск без VLM
%cd /kaggle/working/ai-chemp-cu-itmo
!python search.py --query 'фиона с шреком едят рагу' --db_dir db --data_dir data --mode fused --top_k 5

## (Опционально) VLM через OpenRouter

Если хотите включить VLM rerank в Kaggle, добавьте секрет `OPENROUTER_API_KEY` в **Add-ons -> Secrets**.

In [ ]:
# Опционально подхватываем OPENROUTER_API_KEY из Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['OPENROUTER_API_KEY'] = UserSecretsClient().get_secret('OPENROUTER_API_KEY')
    print('OPENROUTER_API_KEY loaded from Kaggle Secrets')
except Exception as e:
    print('OPENROUTER_API_KEY not loaded:', e)

# Пример поиска с VLM через OpenRouter
# !python search.py --query 'фиона с шреком едят рагу' --db_dir db --data_dir data --mode fused --top_k 5 \
#   --vlm --vlm_provider openrouter --vlm_model qwen/qwen3.5-35b-a3b

## Evaluation (как в eval.ipynb)

Считает retrieval-метрики на `tests/eval_dataset.json`: `Recall@K`, `MRR`, `nDCG`, `mean tIoU`, `mean offset (sec)` и per-query таблицу.

In [ ]:
%cd /kaggle/working/ai-chemp-cu-itmo
import json
import logging
from pathlib import Path
import pandas as pd

from src.embed import load_db, load_model
from eval.dataset import EvalDataset
from eval.evaluate import Evaluator

EVAL_DATASET_PATH = Path('tests/eval_dataset.json')
TOP_K = 5
OVERLAP_THRESHOLD = 0.3
DEVICE = 'cpu'

assert EVAL_DATASET_PATH.exists(), f'Missing eval dataset: {EVAL_DATASET_PATH}'
dataset = EvalDataset.load(EVAL_DATASET_PATH)
print(dataset)

vis_idx, txt_idx, metadata = load_db('db')
model, proc = load_model(
    device=DEVICE,
    cache_dir='/kaggle/working/.model_cache/hf',
    local_files_only=False,
)

logging.basicConfig(level=logging.INFO, format='%(message)s', force=True)

evaluator = Evaluator(
    vis_index=vis_idx,
    txt_index=txt_idx,
    metadata=metadata,
    model=model,
    processor=proc,
    device=DEVICE,
    top_k=TOP_K,
)

results = evaluator.evaluate(
    dataset,
    top_k=TOP_K,
    overlap_threshold=OVERLAP_THRESHOLD,
    judge=False,
)

print('\n' + '='*60)
print(results)
print('='*60)

In [ ]:
if results.retrieval:
    df = pd.DataFrame(results.retrieval.per_query)
    df = df.sort_values(['recall', 'tiou'], ascending=[True, True])

    display_cols = ['query', 'hit', 'recall', 'rr', 'ndcg', 'tiou', 'offset']
    styled = df[display_cols].style.format({
        'recall': '{:.2f}',
        'rr': '{:.2f}',
        'ndcg': '{:.2f}',
        'tiou': '{:.3f}',
    }).map(
        lambda v: 'background-color: #ffcccc' if v == 'MISS' else '',
        subset=['hit']
    )
    styled
else:
    print('Нет retrieval-метрик (нет вопросов с time_spans)')

In [ ]:
out = {
    'config': {
        'top_k': TOP_K,
        'overlap_threshold': OVERLAP_THRESHOLD,
        'dataset_path': str(EVAL_DATASET_PATH),
    }
}

if results.retrieval:
    out['retrieval'] = {
        'recall_at_k': results.retrieval.recall_at_k,
        'mrr': results.retrieval.mrr,
        'ndcg': results.retrieval.ndcg,
        'mean_tiou': results.retrieval.mean_tiou,
        'mean_offset_sec': results.retrieval.mean_offset_sec,
        'total_queries': results.retrieval.total_queries,
        'per_query': results.retrieval.per_query,
    }

Path('notebooks').mkdir(exist_ok=True)
OUTPUT_PATH = Path('notebooks/eval.json')
with OUTPUT_PATH.open('w', encoding='utf-8') as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print('Saved:', OUTPUT_PATH.resolve())

## Повторный запуск без повторной загрузки моделей

После первичного прогона можно использовать флаг `--offline_models`:

```bash
python build_vectordb.py --offline_models --hf_cache_dir /kaggle/working/.model_cache/hf --whisper_cache_dir /kaggle/working/.model_cache/whisper
python search.py --offline_models --hf_cache_dir /kaggle/working/.model_cache/hf --query '...'
```